In [31]:
# Install
!pip install torch torchvision

In [32]:
# Import
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

In [33]:
# Setup environment
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12355"

print("Starting DDP.")


Starting DDP.


In [34]:
# Initialize process group (single process)
dist.init_process_group(
    backend="gloo",
    rank=0,
    world_size=1
)

print("Process group initialized")

Process group initialized


In [35]:
# Simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)

    def forward(self, x):
        return self.fc(x)

model = SimpleModel()
model = DDP(model)

print("Model wrapped with DDP")

Model wrapped with DDP


In [36]:
# Dummy data
x = torch.randn(100, 10)
y = torch.randn(100, 1)

dataset = torch.utils.data.TensorDataset(x, y)

sampler = torch.utils.data.distributed.DistributedSampler(
    dataset,
    num_replicas=1,
    rank=0
)

dataloader = torch.utils.data.DataLoader(dataset, batch_size=10, sampler=sampler)

print("DataLoader ready")

DataLoader ready


In [37]:
# Training function
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

for epoch in range(3):
    sampler.set_epoch(epoch)
    epoch_loss = 0   # track loss

    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        output = model(batch_x)
        loss = loss_fn(output, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch}, Loss {epoch_loss:.4f}", flush=True)

Epoch 0, Loss 16.5835
Epoch 1, Loss 14.6574
Epoch 2, Loss 13.4509


In [38]:
# Cleanup
dist.destroy_process_group()

print("Training Finished.")

Training Finished.
